# 📊 Baseline vs `scale_pos_weight` XGBoost

---

## 1️⃣ Baseline Model (No Class Weighting)

### Confusion Matrix
```
TN = 1999
FP = 85
FN = 155
TP = 227
```

### Minority Class (True)
- **Precision:** 0.73  
- **Recall:** 0.59  
- **F1-score:** 0.65  

### AUC Metrics
- **ROC-AUC:** 0.930  
- **PR-AUC:** 0.747  

### Interpretation

- High overall accuracy (0.90)  
- Good precision for the positive class  
- Misses 155 positive cases (lower recall)  
- Strong ranking ability  

This model is **conservative** when predicting positives.

---

## 2️⃣ XGBoost with `scale_pos_weight`

### Confusion Matrix
```
TN = 1804
FP = 280
FN = 58
TP = 324
```

### Minority Class (True)
- **Precision:** 0.54  
- **Recall:** 0.85  
- **F1-score:** 0.66  

### AUC Metrics
- **ROC-AUC:** 0.930  
- **PR-AUC:** 0.743  

### Interpretation

- Accuracy drops (expected when prioritizing recall)  
- Recall improves significantly (0.59 → 0.85)  
- False negatives reduced (155 → 58)  
- False positives increase (85 → 280)  
- Ranking ability remains unchanged  

This model is **aggressive** in predicting positives.

---

# 🎯 What Actually Changed?

- ROC-AUC ≈ identical  
- PR-AUC ≈ identical  
- Model ranking quality did **not** improve  

What changed is the **decision boundary at threshold = 0.5**.

`scale_pos_weight` shifts predicted probabilities, leading to more positive predictions.

This is a **threshold effect**, not a ranking improvement.

---

# 🔎 Business-Based Model Selection

### If Missing Positives Is Costly  
(e.g., fraud detection, medical diagnosis)

→ Choose the `scale_pos_weight` model  
Because **FN dropped from 155 → 58**

---

### If False Positives Are Expensive  

→ Choose the baseline model  
Because **FP is much lower (85 vs 280)**

---

# ⚠️ Important Engineering Insight

Since ranking metrics are almost identical:

You can likely achieve similar recall improvement by **lowering the classification threshold** in the baseline model — without using `scale_pos_weight`.

Example:

```python
y_pred_custom = (y_proba >= 0.30).astype(int)
```

This allows:
- Controlled recall increase  
- Precision tradeoff management  
- Same ranking quality  

---

# 🏆 Professional Recommendation

1. Keep the **baseline model**  
2. Tune the **classification threshold**  
3. Select threshold using:
   - F1-score  
   - Recall target  
   - MCC  
   - Business cost analysis  

Use `scale_pos_weight` mainly when:
- Imbalance is extreme  
- Or threshold tuning alone cannot achieve required recall  

---

# ✅ Final Comparison Summary

| Metric | Baseline | Weighted |
|--------|----------|----------|
| ROC-AUC | 0.930 | 0.930 |
| PR-AUC | 0.747 | 0.743 |
| Recall (Positive) | 0.59 | **0.85** |
| Precision (Positive) | **0.73** | 0.54 |
| False Negatives | 155 | **58** |
| False Positives | **85** | 280 |

---

## Final Conclusion

The difference is in **classification behavior**, not in underlying ranking quality.

The optimal choice depends entirely on business cost trade-offs.